# 08 — Elyra：就緒檢查 + 取當日輸入（fetch／stage）

**不是** `seed-data.sh`（不會從筆電 `oc cp`）。權重／`bin/` 仍須事先 seed 一次。

| 環境變數 | 預設 | 說明 |
|----------|------|------|
| `DATE` | `20260707` | 測試日 |
| `SHOME` | `/mnt/corrdiff` | PVC 掛載點 |
| `INPUT_MODE` | `grib` | `existing` = 已有 CorrdiffInput；`url` = 下載；`grib` = 準備給下一步 preprocess |
| `INPUT_URL` | （空） | `INPUT_MODE=url` 時的 CorrdiffInput HTTP(S) 位址 |

下一步：`09-elyra-preprocess.ipynb`（`existing`／`url` 時該步會 skip）。


In [ ]:
import os
import urllib.request
from pathlib import Path

SHOME = os.environ.get("SHOME", "/mnt/corrdiff")
DATE = os.environ.get("DATE", "20260707")
INPUT_MODE = os.environ.get("INPUT_MODE", "grib").strip().lower()
INPUT_URL = os.environ.get("INPUT_URL", "").strip()

workdir = Path(SHOME) / "workdir" / DATE
raw_dir = Path(SHOME) / "dat" / "EC_S2S" / DATE
input_nc = workdir / f"CorrdiffInput_EC_RAW_{DATE}.nc"
marker = workdir / ".elyra_input_mode"

print(f"SHOME={SHOME} DATE={DATE} INPUT_MODE={INPUT_MODE}")

# One-time assets (instructor seed-data) — never re-copied here
for rel in ("bin", "config", "etc"):
    p = Path(SHOME) / rel
    ok = p.exists()
    print(f"[{'OK' if ok else 'MISS'}] {p}")
    assert ok, (
        f"Missing {p}. Run instructor seed-data.sh once "
        "(see docs/seed-scripts.md); this step only stages daily input."
    )

workdir.mkdir(parents=True, exist_ok=True)


In [ ]:
assert INPUT_MODE in ("existing", "url", "grib"), f"Bad INPUT_MODE={INPUT_MODE}"

if INPUT_MODE == "existing":
    assert input_nc.is_file(), f"INPUT_MODE=existing but missing {input_nc}"
    print(f"[OK] reuse input {input_nc} ({input_nc.stat().st_size} bytes)")

elif INPUT_MODE == "url":
    assert INPUT_URL, "INPUT_MODE=url requires env INPUT_URL"
    if input_nc.is_file():
        bak = Path(str(input_nc) + ".pre-url-backup.nc")
        bak.write_bytes(input_nc.read_bytes())
        print(f"[OK] backed up previous input -> {bak.name}")
    print(f"Downloading {INPUT_URL} -> {input_nc}")
    urllib.request.urlretrieve(INPUT_URL, input_nc)
    assert input_nc.is_file() and input_nc.stat().st_size > 0
    print(f"[OK] downloaded {input_nc.stat().st_size} bytes")

elif INPUT_MODE == "grib":
    assert raw_dir.is_dir(), f"Missing GRIB dir {raw_dir} (seed-grib-smoke or place C2F.* files)"
    files = sorted(p.name for p in raw_dir.iterdir() if p.is_file())
    assert files, f"No files under {raw_dir}"
    print(f"[OK] GRIB files ({len(files)}): {files[:5]}{'...' if len(files) > 5 else ''}")
    if input_nc.is_file() and not Path(str(input_nc) + ".full-backup.nc").is_file():
        bak = Path(str(input_nc) + ".full-backup.nc")
        bak.write_bytes(input_nc.read_bytes())
        print(f"[OK] backed up CorrdiffInput -> {bak.name} (preprocess may overwrite)")

marker.write_text(INPUT_MODE + "\n")
print(f"[OK] wrote marker {marker} -> {INPUT_MODE}")
print("fetch / stage completed successfully.")
